# Download entirety of the Datasets

In [1]:
import os
import numpy as np
import pandas as pd
import requests
import yfinance as yf

In [2]:
# Remove start_date / end_date (we will pull max history)
output_dir = "../data/sp500_individual_gbm/"
os.makedirs(output_dir, exist_ok=True)

MIN_YEARS = 40
MIN_TRADING_DAYS = MIN_YEARS * 252  # simple rule-of-thumb

In [3]:
import re

def get_sp500_tickers(exclude_problematic=True):
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    table = pd.read_html(response.text)
    df = table[0]

    tickers = df["Symbol"].astype(str).tolist()

    if exclude_problematic:
        # Keep only tickers like AAPL, BRK, GOOG, etc. (no '.', '-', '/', etc.)
        tickers = [t for t in tickers if re.fullmatch(r"[A-Z0-9]+", t)]

    return tickers

In [4]:
def transformation(data):
        # YFinance indexes by Date automatically; ensure it is sorted
        data = data.sort_index()
        
        # Calculate log-returns: r_t = ln(P_t / P_{t-1}) [cite: 5353]
        ohlc_cols = ['Open', 'High', 'Low', 'Close']
        # Divide OHLC by previous day's close to get relative returns
        prev_close = data['Close'].shift(1)
        ohlc_log_rets = np.log(data[ohlc_cols].div(prev_close, axis=0))
        
        # Volume log-returns (adding 1 to avoid log(0))
        volume_log_rets = np.log(data['Volume'] + 1) - np.log(data['Volume'].shift(1) + 1)

        processed_data = pd.concat([ohlc_log_rets, volume_log_rets], axis=1).dropna()
        data_arr = processed_data.values.astype(np.float32)

        # Global Standardization: r_std = (r - mu) / sigma [cite: 5370]
        mean = data_arr.mean(axis=0)
        std = data_arr.std(axis=0)
        data_standardized = (data_arr - mean) / (std + 1e-8)
        
        return data_standardized

In [5]:
tickers = get_sp500_tickers()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_28280\631958006.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(response.text)


In [6]:
tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVB',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BK',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BLDR',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'CPB',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CIEN',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COIN',
 'CL',
 'CMCSA',
 'FIX',
 'CAG',
 'COP',
 'ED',
 'STZ',


In [7]:
tickers = get_sp500_tickers()

for ticker in tickers:
    try:
        df = yf.download(
            tickers=ticker,
            period="max",      # <- full available history
            interval="1d",     # <- daily
            auto_adjust=True,
            progress=False
        )

        if df.empty:
            continue

        # Drop the Ticker level from columns if it exists (yfinance multi-index)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel('Ticker')

        # Basic sanity: ensure sorted index, remove duplicates
        df = df.sort_index()
        df = df[~df.index.duplicated(keep="first")]

        # Filter for long histories (> 40 years)
        span_years = (df.index[-1] - df.index[0]).days / 365.25
        if span_years < MIN_YEARS or len(df) < MIN_TRADING_DAYS:
            print(f"Skipping {ticker}: only {span_years:.1f} years, {len(df)} rows")
            continue

        # Reuse your transformation function
        data_standardized = transformation(df)

        # Save individually (same format as before)
        columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        df_processed = pd.DataFrame(data_standardized, columns=columns)

        first_day = df.index[0].strftime("%Y-%m-%d")
        last_day  = df.index[-1].strftime("%Y-%m-%d")
        file_path = os.path.join(output_dir, f"{ticker}_{first_day}_{last_day}_processed.csv")
        df_processed.to_csv(file_path, index=False)

    except Exception as e:
        print(f"Failed to process {ticker}: {e}")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_28280\631958006.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(response.text)


Skipping ABBV: only 13.2 years, 3316 rows
Skipping ACN: only 24.6 years, 6196 rows
Skipping ADBE: only 39.6 years, 9969 rows
Skipping AES: only 34.7 years, 8738 rows
Skipping A: only 26.3 years, 6615 rows
Skipping ABNB: only 5.2 years, 1316 rows
Skipping AKAM: only 26.4 years, 6629 rows
Skipping ALB: only 32.0 years, 8065 rows
Skipping ARE: only 28.8 years, 7241 rows
Skipping ALGN: only 25.1 years, 6314 rows
Skipping ALLE: only 12.3 years, 3094 rows
Skipping ALL: only 32.8 years, 8248 rows
Skipping GOOGL: only 21.6 years, 5423 rows
Skipping GOOG: only 21.6 years, 5422 rows
Skipping AMZN: only 28.8 years, 7249 rows
Skipping AMCR: only 13.8 years, 3474 rows
Skipping AEE: only 28.2 years, 7089 rows
Skipping AMT: only 28.0 years, 7051 rows
Skipping AWK: only 17.9 years, 4498 rows
Skipping AMP: only 20.5 years, 5151 rows
Skipping APH: only 34.3 years, 8643 rows
Skipping APO: only 14.9 years, 3758 rows
Skipping APP: only 4.9 years, 1231 rows
Skipping APTV: only 14.3 years, 3596 rows
Skipping